so here the core concept was first we should create a brain(network), this one is nn model that gets the state and gave us the Q value.

then we have two model, one target model that calcuate future value, and other one is the main model that value for current state,action.(this act as table that we have)

then we have optimize function to update our network based on these two models. and done


In [62]:
import numpy as np
import gymnasium as gym
import ale_py
from collections import deque, Counter
import random
from datetime import datetime
import torch
from torch import nn


In [81]:
env = gym.make("MsPacman-v0")

color = np.array([210, 164, 74]).mean()
n_outputs = env.action_space.n

epsilon = 0.5
eps_min = 0.05
eps_max = 1.0
eps_decay_steps = 500000

buffer_len = 20000
exp_buffer = deque(maxlen=buffer_len)


num_episodes = 800
batch_size = 32
input_shape = (None, 88, 80, 1)
learning_rate = 0.001
X_shape = (None, 88, 80, 1)
discount_factor = 0.97

global_step = 0
copy_steps = 100
steps_train = 4
start_steps = 2000


In [82]:



def preprocess_observation(obs):

    # Crop and resize the image
    img = obs[1:176:2, ::2]

    # Convert the image to greyscale
    img = img.mean(axis=2)

    # Improve image contrast
    img[img==color] = 0

    # Next we normalize the image from -1 to +1
    img = (img - 128) / 128 - 1

    return img.reshape(88,80,1)

Output=(
Input+2×Padding−Kernel Size
​
 )/stride+1

In [83]:


class Q_network(nn.Module):
    def __init__(self, num_actions):
        super(Q_network, self).__init__()

        self.conv2d1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=8, stride=4, padding=2) # Output: (32, 22, 20)
        self.conv2d2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2, padding=1) #➡️ Output: (64, 11, 10)
        self.conv2d3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1) #➡️ Output: (64, 11, 10)

        self.flatten = nn.Flatten()

        flattened_size = 64 * 10 * 10

        self.fc1 = nn.Linear(7040, 512)
        self.fc_out = nn.Linear(512, num_actions)

    def forward(self, x):
        x = torch.relu(self.conv2d1(x))
        x = torch.relu(self.conv2d2(x))
        x = torch.relu(self.conv2d3(x))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        return self.fc_out(x)


In [84]:

def epsilon_greedy(action, step):
    epsilon = max(eps_min, eps_max - (eps_max-eps_min) * step/eps_decay_steps)
    if np.random.rand() < epsilon:
        return np.random.randint(n_outputs)
    else:
        return action

In [85]:
main_Q = Q_network(n_outputs)
target_Q = Q_network(n_outputs)

target_Q.load_state_dict(main_Q.state_dict())


<All keys matched successfully>

In [86]:
def optimize_model():

    if len(exp_buffer) < batch_size:
        return

    batch = random.sample(exp_buffer,batch_size)

    state_batch, action_batch, reward_batch, next_state_batch, done_batch = zip(*batch)

    state_batch = torch.FloatTensor(state_batch)
    action_batch = torch.LongTensor(action_batch).unsqueeze(1)
    reward_batch = torch.FloatTensor(reward_batch)
    next_state_batch = torch.FloatTensor(next_state_batch)
    done_batch = torch.FloatTensor(done_batch)

    q_values = main_Q(state_batch).gather(1,action_batch).squeeze()

    with torch.no_grad():
        q_next_max = target_Q(next_state_batch).max(1)[0]
        target_q_value = reward_batch + discount_factor * q_next_max * ( 1 - done_batch)

        
    loss = nn.MSELoss()(q_values, target_q_values)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    
    

In [88]:
for i in range(2):

    done = False
    obs = env.reset()[0]
    epoch = 0
    episodic_reward = 0
    actions_counter = Counter() 
    episodic_loss = []
    counter = 0
    while not done:

        obs = preprocess_observation(obs) # state or picture
        #print(obs.shape)
        obs_tensor = torch.tensor(obs,dtype=torch.float32)
        obs_tensor = obs_tensor.permute(2, 0, 1)
        #print(obs_tensor.shape)
        obs_tensor = obs_tensor.unsqueeze(0)
        q_values = main_Q(obs_tensor)
        action = torch.argmax(q_values).item() # we did argmax to get right action and not value


        # select the action using epsilon greedy policy add exploration on that
        action = epsilon_greedy(action, global_step)
        
        next_obs,reward,done,_,info = env.step(action)
        
        next_state =  preprocess_observation(next_obs)
        next_state_tensor = torch.tensor(next_state,dtype=torch.float32)
        next_state_tensor = next_state_tensor.permute(2, 0, 1)
        
        exp_buffer.append([obs_tensor, action,next_state_tensor, reward, done])

        optimize_model()

        obs = next_obs
        
        if counter % 100 ==0:
            target_Q.load_state_dict(main_Q.state_dict())

        counter +=1
    epoch +=1
        

        
        

ValueError: only one element tensors can be converted to Python scalars

In [20]:
len(env.reset())

2

In [27]:
obs = env.reset()[0]
obs = preprocess_observation(obs)

In [28]:
obs.shape

(88, 80, 1)

In [34]:
obs_tensor = torch.tensor(obs, dtype=torch.float32)

# Permute from (H, W, C) to (C, H, W) => expected by Conv2d
obs_tensor = obs_tensor.permute(2, 0, 1)

# Add batch dimension: (1, C, H, W)
obs_tensor = obs_tensor.unsqueeze(0)

# Now pass it to the model
actions = main_Q(obs_tensor)
action = torch.argmax(actions).item() # we did argmax to get right action and not value


In [35]:
action

2

In [39]:
env.step(action)

(array([[[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        [[228, 111, 111],
         [228, 111, 111],
         [228, 111, 111],
         ...,
         [228, 111, 111],
         [228, 111, 111],
         [228, 111, 111]],
 
        [[228, 111, 111],
         [228, 111, 111],
         [228, 111, 111],
         ...,
         [228, 111, 111],
         [228, 111, 111],
         [228, 111, 111]],
 
        ...,
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
  